In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error
import optuna
from sklearn.ensemble import RandomForestRegressor
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.preprocessing import StandardScaler
import joblib
import json

# Functions

In [2]:
def naive_forecaster(X_test):
    lag1_cols = [f'DA_price lag1_hour{i}' for i in range(24)]
    lag7_cols = [f'DA_price lag7_hour{i}' for i in range(24)]
    prediction = []
    for index, row in X_test.iterrows():
        if pd.to_datetime(index).weekday() in [1,2,3,4,6]:
            prediction.append(row[lag1_cols].values)
        else:
            prediction.append(row[lag7_cols].values)
    prediction = np.array(prediction)
    return prediction

# calculate smape
def smape(y_true, y_pred):
    return 100/len(y_true) * np.sum(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

#day average error
def dae(y_true, y_pred):
    return np.mean(np.abs(np.mean(y_pred,axis=1) - np.mean(y_true,axis=1)))

def rmae(y_true, y_pred, y_pred_naive):
    mae_pred = np.mean(np.abs(y_pred - y_true))
    mae_naive = np.mean(np.abs(y_pred_naive - y_true))
    return mae_pred/mae_naive

In [4]:
def get_best_params_and_test(country, period=0, n_trials=50):

    # Load the data
    inputs = pd.read_csv(os.path.join("cut_data", country, "inputs.csv"),index_col=0)
    inputs.fillna(0, inplace=True)
    outputs = pd.read_csv(os.path.join("cut_data", country, "outputs.csv"),index_col=0)
    outputs.fillna(value=0, inplace=True)

    X_train = inputs.loc[f'{2015+period}-01-09':f'{2018+period}-12-31']
    y_train = outputs.loc[f'{2015+period}-01-09':f'{2018+period}-12-31']

    X_val = inputs.loc[f'{2019+period}-01-01':f'{2019+period}-12-31']
    y_val = outputs.loc[f'{2019+period}-01-01':f'{2019+period}-12-31']

    X_test = inputs.loc[f'{2020+period}-01-01':f'{2020+period}-12-31']
    y_test = outputs.loc[f'{2020+period}-01-01':f'{2020+period}-12-31']

    # Scaling
    scaler1, scaler2 = StandardScaler(), StandardScaler()

    X_train = pd.DataFrame(scaler1.fit_transform(X_train), columns=X_train.columns)
    X_val = pd.DataFrame(scaler1.transform(X_val), columns=X_val.columns)
    X_test = pd.DataFrame(scaler1.transform(X_test), columns=X_test.columns)

    y_train = pd.DataFrame(scaler2.fit_transform(y_train), columns=y_train.columns)
    y_val = pd.DataFrame(scaler2.transform(y_val), columns=y_val.columns)
    y_test = pd.DataFrame(scaler2.transform(y_test), columns=y_test.columns)

    # Generate the list of column names
    column_names = [f'DA_price lag1_hour{i}' for i in range(24)]

    # Get the indices of the columns
    column_indices = [X_train.columns.get_loc(name) for name in column_names if name in X_train.columns]


    # transform y_train_val, y_test by substraction of the X_train_val X_test column_indices values
    y_train2 = pd.DataFrame(y_train.values - X_train.iloc[:,column_indices].values, columns=y_train.columns, index=y_train.index)
    y_val2 = pd.DataFrame(y_val.values - X_val.iloc[:,column_indices].values, columns=y_val.columns, index=y_val.index)
    y_test2 = pd.DataFrame(y_test.values - X_test.iloc[:,column_indices].values, columns=y_test.columns, index=y_test.index)
    
    y_val_val = y_val.values
    def objective(trial):
        # Define the hyperparameters to tune
        n_estimators = trial.suggest_int('n_estimators', 10, 500)
        max_depth = trial.suggest_int('max_depth', 2, 32)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
        max_features = trial.suggest_categorical('max_features', [None, 'sqrt', 'log2'])

        # Define the RandomForestRegressor model with the suggested hyperparameters
        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=42,
            n_jobs=-1
        )
    
        # Fit the model
        model.fit(X_train, y_train2)
    
        # Predict on validation set
        y_pred2 = model.predict(X_val)
        y_pred = y_pred2 + X_val.values[:, column_indices]
    
        # Flatten the predicted and actual values
        y_pred_flattened = y_pred.flatten()
        y_val_flattened = y_val.values.flatten()
        y_pred_naive_flattened = naive_forecaster(X_val).flatten()
    
        # Calculate mean absolute error between flattened vectors
        rmae_val = rmae(y_val_flattened, y_pred_flattened, y_pred_naive_flattened)
    
        # Return the MAE as the objective to minimize
        return rmae_val
    
    # Create an Optuna study
    study = optuna.create_study(direction='minimize')
    
    # Optimize the hyperparameters
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)  # You can adjust the number of trials
    
    model = RandomForestRegressor(**study.best_params, random_state=42, n_jobs=-1)

    X_train_val = pd.concat([X_train, X_val])
    y_train_val2 = pd.concat([y_train2, y_val2])

    model.fit(X_train_val, y_train_val2)
    
    y_pred2 = model.predict(X_test)
    y_pred = y_pred2 + X_test.values[:, column_indices]

    y_pred_naive = naive_forecaster(X_test)
    y_test_val = y_test.values

    # inverse transform y_pred, y_pred_naive, y_test_val
    y_pred = scaler2.inverse_transform(y_pred)
    y_pred_naive = scaler2.inverse_transform(y_pred_naive)
    y_test_val = scaler2.inverse_transform(y_test_val)
    
    smape_score = smape(y_test_val.flatten(), y_pred.flatten())
    mae_score = mean_absolute_error(y_test_val.flatten(), y_pred.flatten())
    dae_score = dae(y_test_val, y_pred)
    rmae_score = rmae(y_test_val, y_pred, y_pred_naive)
    
    naiv_smape = smape(y_test_val.flatten(), y_pred_naive.flatten())
    naiv_mae = mean_absolute_error(y_test_val.flatten(), y_pred_naive.flatten())
    naiv_dae =  dae(y_test_val, y_pred_naive)
    
    
    return [country, period, smape_score, mae_score, dae_score, rmae_score, naiv_smape, naiv_mae, naiv_dae, str(study.best_params)]

# Training and evaluation

In [ ]:
country_name_list = sorted(os.listdir(os.path.join('cut_data')))

# load final results
#with open('rf_final_results.json', 'r') as f:
#    final_results = json.load(f)
final_results = {}

for country in tqdm(country_name_list):
    for period in [0,4]:
        results = get_best_params_and_test(country, period, n_trials=50)
        final_results[f"{country}_{period}"] = results
        # save as json
        with open('rf_final_results.json', 'w') as f:
            json.dump(final_results, f)
    #print(f'{country} is done')

100%|██████████| 5/5 [1:47:52<00:00, 1294.56s/it]


## Save results as csv file

In [7]:
import json
import numpy as np
import pandas as pd

In [8]:
with open('rf_final_results.json', 'r') as f:
    final_results = json.load(f)

In [9]:
df = pd.DataFrame(final_results).T
df.columns = ['country', 'period', 'smape', 'mae', 'dae', 'rmae', 'naive_smape', 'naive_mae', 'naive_dae', 'best_params']
df.index = np.arange(len(df))
#df.drop(columns=['best_params'], inplace=True)
df.to_csv('rf_final_results.csv', index=False)